# Filtered surrogate scans

Removes failed or numerically corrupted simulations **before** fitting the surrogate (`iceburner.surrogate.filter_invalid_simulations`), refits the output scaler, and trains the random-forest surrogate. Then scans outer-radius, DT ice-thickness, Be-thickness, current, and gas-density inputs.

In [ ]:
import sys
sys.path.insert(0, "..")

import copy

import numpy as np
import millefeuille as mf
from iceburner import surrogate

state_raw = mf.State.load("../data/Gorgon_MF_DB_Gorgon_MF_iceburner_final_week.db")
state = copy.deepcopy(state_raw)

state, removed_df = surrogate.filter_invalid_simulations(state)
removed_df

In [ ]:
surrogate_model = mf.MultiOutputRandomForestSurrogate(
    n_outputs=len(state.Y_names),
    verbose=True,
)
surrogate_model.fit(state)

In [ ]:
X_train = np.asarray(state.Xs, dtype=float)

print("Input names:")
for i, name in enumerate(state.X_names):
    print(f"  {i}: {name}")

print("\nObjective names:")
for i, name in enumerate(state.Y_names):
    print(f"  {i}: {name}")

print("\nTraining-data ranges:")
for i, name in enumerate(state.X_names):
    print(
        f"{name:20s}: "
        f"{np.nanmin(X_train[:, i]):.6g} to "
        f"{np.nanmax(X_train[:, i]):.6g}"
    )

## Outer-radius multiplier scan

In [ ]:
radius_scan = surrogate.scan_input(
    state, surrogate_model, "R_outer", X_train=X_train,
    scan_min=0.9, scan_max=1.1, n_points=100,
    xlabel=r"$f_{R_{\mathrm{outer}}}$",
)

## DT ice-thickness scan

DT ice thickness is derived rather than stored directly as a surrogate input. This scan varies `f_Pi`, computes the corresponding physical DT ice thickness (`iceburner.surrogate.scan_dt_ice_thickness`, backed by `iceburner.scaling.dt_ice_geometry`), and plots every objective against ice thickness.

In [ ]:
ice_scan = surrogate.scan_dt_ice_thickness(
    state, surrogate_model, X_train=X_train,
    pi_min=0.98, pi_max=1.02, n_points=100,
)

## Optional Be-thickness scan

This scans the Be liner thickness, which is different from DT ice thickness.

In [ ]:
be_scan = surrogate.scan_input(
    state, surrogate_model, "Be", X_train=X_train,
    scan_min=0.5, scan_max=1.5, n_points=100,
    xlabel=r"$f_{\mathrm{Be}}$",
)

In [ ]:
current_scan = surrogate.scan_input(
    state, surrogate_model, "current", X_train=X_train,
    scan_min=20, scan_max=60, n_points=200,
    xlabel=r"$f_{\mathrm{current}}$",
)

In [ ]:
rho_scan = surrogate.scan_input(
    state, surrogate_model, "f_rho", X_train=X_train,
    scan_min=0.5, scan_max=1.5, n_points=100,
    xlabel=r"$f_{\mathrm{f_rho}}$",
)

## Gas-density scan at several fixed currents

In [ ]:
current_idx = surrogate.find_name_index(state.X_names, "current")

for current_ma in [20, 40, 60]:
    anchor = surrogate.make_anchor(X_train)
    anchor[current_idx] = current_ma

    surrogate.scan_input(
        state, surrogate_model, "f_rho", X_train=X_train,
        scan_min=0.5, scan_max=1.5, n_points=100,
        anchor=anchor, xlabel=r"$f_{\rho}$",
    )